In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Runtime requirements for EvoSuite test generation per project

In [ ]:
import pandas as pd

query = """
SELECT 
    rt.*,
    project_name(rt.project_id) AS project_name
FROM 
    mv_evosuite_runtime_pivoted rt
"""

df = pd.read_sql(query, conn)

# Define the runtime columns we want to analyze
runtime_columns = [
    'total', 'search', 'inlining', 'minimization', 'coverage_analysis', 
    'assertion_generation', 'junit_check', 'writing_tests', 
    'writing_statistics', 'done', 'finished'
]

# Create a dataframe with total runtime per project + phase
project_runtime_df = df.groupby(['project_id', 'project_name'])[runtime_columns].sum().reset_index()

# Sort by project_id to match the SQL view
project_runtime_df = project_runtime_df.sort_values('project_id')

# Display the results
print("Total EvoSuite runtime by project:")
display(project_runtime_df)

## Average runtime requirements for EvoSuite test generation per search budget + class

In [ ]:
import pandas as pd

query = """
SELECT 
    mv.*,
    p.configuration::json->'evosuite'->>'search-budget' AS search_budget
FROM 
    mv_evosuite_runtime_pivoted mv
JOIN 
    project p ON mv.project_id = p.id
"""

df = pd.read_sql(query, conn)

# Convert search_budget to numeric
df['search_budget'] = pd.to_numeric(df['search_budget'], errors='coerce')

# Define the runtime columns we want to analyze
runtime_columns = [
    'total', 'search', 'inlining', 'minimization', 'coverage_analysis', 
    'assertion_generation', 'junit_check', 'writing_tests', 
    'writing_statistics', 'done', 'finished'
]

# Create mean dataframe
mean_df = df.groupby('search_budget')[runtime_columns].mean().reset_index()
mean_df = mean_df.sort_values('search_budget')

# Create median dataframe
median_df = df.groupby('search_budget')[runtime_columns].median().reset_index()
median_df = median_df.sort_values('search_budget')

# Display the results
print("Mean EvoSuite runtimes per class:")
display(mean_df)
print("Median EvoSuite runtimes per class:")
display(median_df)

# Calculate percentage of time spent in each phase
percentage_columns = [
    'search', 'inlining', 'minimization', 'coverage_analysis', 
    'assertion_generation', 'junit_check', 'writing_tests', 
    'writing_statistics', 'done', 'finished'
]

# Create a new dataframe for percentages
df_percentages = df.copy()

# Calculate percentages for each phase relative to total
for col in percentage_columns:
    df_percentages[f'{col}_pct'] = (df_percentages[col] / df_percentages['total']) * 100

# Get percentage columns
pct_columns = [f'{col}_pct' for col in percentage_columns]

# Create mean percentage dataframe
mean_pct_df = df_percentages.groupby('search_budget')[pct_columns].mean().reset_index()
mean_pct_df = mean_pct_df.sort_values('search_budget')

# Create median percentage dataframe
median_pct_df = df_percentages.groupby('search_budget')[pct_columns].median().reset_index()
median_pct_df = median_pct_df.sort_values('search_budget')

# Display the percentage results
print("\nMean percentage of time spent in each phase:")
display(mean_pct_df)
print("\nMedian percentage of time spent in each phase:")
display(median_pct_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming mean_df is already created and contains the data we need
# We'll transpose the data to get phases as rows and search budgets as columns
# First, set search_budget as index
mean_df_indexed = mean_df.set_index('search_budget')

# Select only the phase columns (exclude 'total' as it's the sum of all phases)
phase_columns = [
    'search', 'inlining', 'minimization', 'coverage_analysis', 
    'assertion_generation', 'junit_check', 'writing_tests', 
    'writing_statistics', 'done', 'finished'
]

# Transpose to get phases as rows and search budgets as columns
plot_data = mean_df_indexed[phase_columns].transpose()

# Create the plot
fig, ax = plt.subplots(figsize=(16, 4))

# Get the number of phases and search budgets
n_phases = len(phase_columns)
n_budgets = len(plot_data.columns)

# Set the width of each bar and the spacing between groups
bar_width = 0.8 / n_budgets
group_spacing = np.arange(n_phases)

# Use tab10 colormap
colors = plt.cm.tab10(np.arange(n_budgets) % 10)

# Find the maximum value to adjust y-axis limits
max_value = plot_data.max().max()

# Plot each search budget as a set of bars
for i, (budget, color) in enumerate(zip(plot_data.columns, colors)):
    positions = group_spacing + (i - n_budgets/2 + 0.5) * bar_width
    bars = ax.bar(positions, plot_data[budget], bar_width, label=f'Budget: {budget}s', color=color)

    # Add text labels on top of each bar
    for bar_idx, bar in enumerate(bars):
        height = bar.get_height()
        # Format value to 1 decimal place if it's >= 1, otherwise 2 decimal places
        if height >= 1:
            value_text = f'{height:.1f}'
        else:
            value_text = f'{height:.2f}'

        # Position the text above the bar
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 1,  # Scale the offset based on the maximum value
            value_text,
            ha='center',
            va='bottom',
            fontsize=8,
        )

# Set the x-axis labels and positions
ax.set_xticks(group_spacing)
ax.set_xticklabels(phase_columns, rotation=45, ha='right')

# Add labels and title
ax.set_ylabel('Mean Runtime (seconds)')
ax.set_title('Mean EvoSuite Runtime by Phase and Search Budget')

# Add a legend
ax.legend(title='Search Budget (seconds)')

# Add grid lines for better readability
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Set y-axis limit to add more space for labels (add 15% padding)
ax.set_ylim(0, max_value * 1.15)

# Adjust layout to make room for labels
plt.tight_layout()

# Show the plot
plt.show()
